# Variant Effect Scanner: Colab scan (Task 1)
Runtime > Change runtime type > **GPU** (T4 is fine). Run the cells in order.
Everything lives on Google Drive, so if Colab disconnects, reconnect and re-run from cell 2: `scan_all.py` skips proteins that are already saved.
Before starting: push the local commits to GitHub (`git push`), otherwise the clone will not have the Task 1 scripts.

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("NO GPU: change the runtime type before continuing")

In [ ]:
# Drive keeps scans safe across disconnects
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_URL = "https://github.com/A-k-s-h-a-t-G-u-p-t-a/Variant-effect-scanner.git"
REPO = "/content/drive/MyDrive/Variant-effect-scanner"
if os.path.isdir(REPO + "/.git"):
    !git -C "{REPO}" pull --ff-only
else:
    !git clone "{REPO_URL}" "{REPO}"
%cd {REPO}
!git log --oneline | head -3
!ls scripts

In [ ]:
# Do NOT use requirements.txt here (it pins a CPU-only torch). Colab already has a CUDA torch.
!pip install -q transformers==5.19.0 scikit-learn scipy pandas tqdm

## GPU vs CPU check (about a minute)
Rescans one protein (CISD1, 108 aa) on the GPU and compares with the cached CPU scan. Must print PASS before the big scan.

In [ ]:
!python scripts/check_device.py --device cuda --acc Q9NZ45

## Scan the remaining proteins
Reads `data/annotated_proteins_all.json` (126 proteins), skips the cached ones (25 + any finished earlier), saves each scan to `results/logprobs/` as soon as it finishes. Safe to interrupt and re-run.

In [ ]:
!python scripts/scan_all.py --device cuda

## Check the result

In [ ]:
import glob, json
import numpy as np
prots = json.load(open("data/annotated_proteins_all.json"))
have = {os.path.basename(p)[:-4] for p in glob.glob("results/logprobs/*.npy")}
missing = [p["accession"] for p in prots if p["accession"] not in have]
print(len(prots) - len(missing), "of", len(prots), "scanned; missing:", missing)
for p in prots:
    if p["accession"] in have:
        a = np.load(f"results/logprobs/{p['accession']}.npy")
        assert a.shape == (len(p["sequence"]), 20) and np.isfinite(a).all(), p["accession"]
print(open("results/scan_meta.json").read())

In [ ]:
# Rebuild the table and the bootstrap AUC on all scanned proteins (about a minute)
!python scripts/build_site_table.py
!python scripts/auc_bootstrap.py

In [ ]:
# Download a zip of the new scans (they are also on Drive in the repo folder)
!zip -qr scans.zip results/logprobs results/scan_meta.json results/site_table.csv results/auc_bootstrap.csv
from google.colab import files
files.download("scans.zip")